# Les files — Exercice 1 : la classe `File` (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** structure de données *file* (FIFO), programmation objet, implémenter une API donnée, déclencher une erreur, méthode spéciale `__repr__`.

Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

---
# QUESTION 1

#### 📝 ÉNONCÉ

> Le langage python ne proposant pas de type « file » au sens strict du terme, il est toutefois possible d'en définir un à l'aide de la programmation orientée objet.
>
> 1. Dans un fichier `file.py`, écrire la définition de la classe `File` dont on donne l'API ci-dessous.
> ```
> class File(builtins.object)
>  |
>  | __init__(self)
>  |      Construit une nouvelle file vide
>  |
>  | enfile(self, el: Object) -> None
>  |      Ajoute un élément à la fin de la file.
>  |
>  | defile(self) -> Object
>  |      Supprime et renvoie l'élément en tête de file.
>  |      Déclenche une erreur si la file est vide.
>  |
>  | taille(self) -> int
>  |      Renvoie le nombre d éléments de la file.
>  |
>  | est_vide(self) -> bool
>  |      Renvoie True si la file est vide, False sinon.
>  |
>  | __repr__(self) -> str
>  |      Renvoie une description du contenu de la file.
> ```
> On pourra s'inspirer de la classe `Pile` et consulter la documentation de la méthode `pop` du type `list`.

#### ✅ RÉSOLUTION

On range les éléments dans une liste python `self.contenu` : la **tête** de la file est au **début** de la liste (indice 0), la **queue** à la **fin**.

La cellule ci-dessous **écrit le fichier** `file.py` (commande `%%writefile` de Jupyter) : c'est le contenu à rendre.

In [1]:
%%writefile file.py
class File:
    """File (queue) en mode FIFO : premier entré, premier sorti.

    Les éléments sont rangés dans une liste python : la tête de la file
    est au début de la liste (indice 0), la queue à la fin.
    """

    def __init__(self):
        """Construit une nouvelle file vide."""
        self.contenu = []

    def enfile(self, el):
        """Ajoute un élément à la fin de la file."""
        self.contenu.append(el)

    def defile(self):
        """Supprime et renvoie l'élément en tête de file.
        Déclenche une erreur si la file est vide."""
        if self.est_vide():
            raise IndexError("defile() sur une file vide")
        return self.contenu.pop(0)

    def taille(self):
        """Renvoie le nombre d'éléments de la file."""
        return len(self.contenu)

    def est_vide(self):
        """Renvoie True si la file est vide, False sinon."""
        return self.taille() == 0

    def __repr__(self):
        """Renvoie une description du contenu de la file :
        les éléments de la tête (à gauche) à la queue (à droite)."""
        return "File(" + repr(self.contenu) + ")"

Overwriting file.py


*Vérification (non demandée) : on importe la classe et on teste le FIFO et le cas d'erreur.*

In [2]:
from file import File

f = File()
print(f, "| taille :", f.taille(), "| vide ?", f.est_vide())
for x in [10, 20, 30]:
    f.enfile(x)
print(f, "| taille :", f.taille(), "| vide ?", f.est_vide())

assert f.defile() == 10          # le premier entré est le premier sorti
assert f.defile() == 20
assert f.defile() == 30
assert f.est_vide()
try:
    f.defile()
    print("ERREUR : aucune exception déclenchée")
except IndexError as e:
    print("Erreur bien déclenchée :", e)

File([]) | taille : 0 | vide ? True
File([10, 20, 30]) | taille : 3 | vide ? False
Erreur bien déclenchée : defile() sur une file vide


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Une file, c'est une file d'attente :** mode **FIFO** (*First In, First Out*), le premier arrivé est le premier servi. On ajoute en queue (`enfile`), on retire en tête (`defile`).

| Méthode | Code | Pourquoi |
|:---|:---|:---|
| `enfile(el)` | `self.contenu.append(el)` | `append` ajoute à la fin de la liste, donc en queue |
| `defile()` | `self.contenu.pop(0)` | `pop(i)` **supprime et renvoie** l'élément d'indice `i` : `pop(0)` renvoie la tête |
| `taille()` | `len(self.contenu)` | |
| `est_vide()` | `self.taille() == 0` | on réutilise une méthode déjà écrite |

- **L'indication de l'énoncé :** la classe `Pile` utilise `pop()` **sans argument**, qui retire le **dernier** élément (LIFO). Pour une file, on retire le **premier** : `pop(0)`. Toute la différence entre pile et file tient dans cet indice.
- **« Déclenche une erreur si la file est vide » :** `pop(0)` sur une liste vide déclencherait déjà une erreur, mais avec un message qui parle de liste (`pop from empty list`) alors qu'on manipule une file. On déclenche donc une erreur claire avec `raise IndexError("...")`.
- **`__repr__`** est une méthode spéciale : Python l'appelle quand on affiche l'objet (`print(f)`). Sans elle, on verrait `<file.File object at 0x7f...>`. Elle doit **renvoyer** une chaîne (`return`), pas l'afficher (`print`).
- **Ce qu'on affiche :** uniquement les éléments, de la tête à la queue. On réutilise l'affichage de la liste `self.contenu` : `repr(self.contenu)` donne `['a', 'b', 'c']`. On ajoute `File(...)` autour pour qu'on ne confonde pas la file avec une liste : `File(['a', 'b', 'c'])`.
- **Pourquoi `repr` ?** La fonction `repr(x)` appelle la méthode `__repr__` de `x`. Par convention, `__repr__` montre une valeur **comme on l'écrit en Python** : une chaîne garde ses apostrophes (`repr("5")` donne `'5'`, alors que `str("5")` donne `5`). L'affichage d'une liste applique `repr` à chacun de ses éléments : on voit donc tout de suite si la file contient le texte `'5'` ou le nombre `5`.
- ⚠️ On n'écrit **pas** de « tête » ou de « queue » dans l'affichage : ce ne sont pas des éléments de la file. La tête est simplement le premier élément affiché, la queue le dernier.
- ⚠️ **Erreurs fréquentes :** `pop()` au lieu de `pop(0)` (on obtient une pile) ; oublier le `return` dans `defile` (l'élément est supprimé mais la méthode renvoie `None`).

---
# QUESTION 2

#### 📝 ÉNONCÉ

> 2. On considère le script ci-dessous.
> ```python
> f = File()
> f.enfile("n")
> f.enfile("s")
> f.enfile("i")
> x = f.defile()
> f.defile()
> f.enfile(x)
> ```
> Donner l'état de la file `f` à l'issue de ce script.

#### ✅ RÉSOLUTION

| Instruction | État de `f` (tête à gauche) | `x` |
|:---|:---|:---|
| `f = File()` | *(vide)* | |
| `f.enfile("n")` | `n` | |
| `f.enfile("s")` | `n s` | |
| `f.enfile("i")` | `n s i` | |
| `x = f.defile()` | `s i` | `"n"` |
| `f.defile()` | `i` | `"n"` |
| `f.enfile(x)` | `i n` | `"n"` |

**À la fin, la file contient `"i"` en tête, puis `"n"` en queue.**

*Vérification (non demandée) :*

In [3]:
f = File()
f.enfile("n")
f.enfile("s")
f.enfile("i")
x = f.defile()
f.defile()
f.enfile(x)
print(f)

File(['i', 'n'])


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Méthode :** suivre le script ligne par ligne dans un tableau, en notant l'état de la file **et** des variables.
- La ligne `f.defile()` (sans `x =`) supprime bien `"s"` : la valeur renvoyée est simplement perdue.
- `"n"`, premier entré, est le premier sorti : c'est le FIFO. Ré-enfilé, il repasse **derrière** `"i"`.